# Theory companion: ML surrogates for a 1T′-WTe₂ topological-insulator FET
**ECE 455/555 course project, Shuvankar Biswas**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKBiswas1998/ML-Project-for-ECE/blob/test_1/notebooks/WTe2_TIFET_theory.ipynb)

This notebook walks through **every component of the pipeline in the order the data flows through it**. Each section has four parts:

1. **What we do**: the step in plain words.
2. **Theory**: the idea and the key equations.
3. **What is optimised**: the exact objective, and what decides its settings.
4. **Results**: figures and tables loaded from the repository (`results/*.csv`, `figures/*.png`).

No result number is written into the text. Every number you see is printed by a code cell from the saved results, so text and data cannot disagree. The companion notebook `WTe2_TIFET_pipeline.ipynb` can re-run the models; this one only reads and explains. Runtime: about 2–3 minutes on Colab or Kaggle (CPU).

## 0. Setup: get the code, data and results from GitHub

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL, BRANCH = "https://github.com/SKBiswas1998/ML-Project-for-ECE.git", "test_1"
REPO = Path("ML-Project-for-ECE").resolve()
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "scikit-learn>=1.5", "psutil"], check=True)
sys.path.insert(0, str(REPO / "python"))
os.environ["PYTHONWARNINGS"] = "ignore"

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
from wte2ml.config import load_config, repo_path
cfg = load_config()
res, fig_dir = repo_path(cfg, "results_dir"), repo_path(cfg, "figures_dir")
def show(name):                      # display one saved figure
    display(Image(str(fig_dir / name)))
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

## 1. The device and the data

**What we do.** We learn from a simulated dataset. A 1T′-WTe₂ nanoribbon (about 5.7 nm × 10.5 nm) sits between a top and a bottom gate. The gate field opens a gap in the topological edge states and switches the current off. Edge atoms are removed at random (edge vacancies), with one random layout in which each higher vacancy level keeps the atoms removed at lower levels and removes a few more (**nested** layouts).

**Theory: how the data were made (NEGF).** The drain current comes from the Landauer formula with a transmission $T(E)$ computed by non-equilibrium Green's functions:

$$ I = \frac{q}{h}\int T(E)\,\big[f_S(E) - f_D(E)\big]\,dE,\qquad f(E) = \frac{1}{1+e^{(E-\mu)/k_BT}} $$

$$ G^R(E) = \big[E - H - \Sigma_S(E) - \Sigma_D(E)\big]^{-1},\qquad T(E) = \mathrm{Tr}\big[\Gamma_S G^R \Gamma_D G^{R\dagger}\big] $$

Temperature enters only through the Fermi functions; vacancies change $H$ (orbitals removed). The run is coherent and non-self-consistent (no Poisson solve).

**Inputs and outputs for ML.** The inputs are temperature $T$ (4–78 K), vacancy level (0–2 %) and gate voltage $V_{TG}-V_{BG}$ (20 points). The outputs are currents and derived device metrics. Width and length never vary, so they are not inputs.

**Why log10.** The off-current spans many orders of magnitude, so a squared error on raw amperes would be dominated by the largest currents. Every current-like target is modelled as $\log_{10}$, and errors are then measured in **decades**.

In [ ]:
from wte2ml.data import load_dataset, curve_matrix, load_layout
ds = load_dataset(cfg)                                   # validates row counts, NaNs, grid, splits
keys, gate_V, Y = curve_matrix(ds.iv_pooled(), cfg)
print("curves:", Y.shape[0], "| gate points per curve:", Y.shape[1], "| nested layouts:", load_layout(cfg).is_nested())
ds.metrics_pooled()[["T_K", "vac_pct", "Ion_A", "Ioff_A", "log10_Ion_Ioff"]].describe().round(3)

**Figure: the headline quantity, log10 Ion/Ioff, over the whole temperature × vacancy grid.**

In [ ]:
show("ion_ioff_heatmap.png")

## 2. Targets: what we predict

**What we do.** We turn each gate sweep into learning targets:
- **log10 I** at each gate point (the whole I–V curve).
- **log10 Ion/Ioff (headline).** Ion is the current at $V=0$ and Ioff the current at the last gate point (1.9 V), as supplied.
- **log10 Ion/Ioff (min)** as a sensitivity check, using the lowest current of the sweep instead. Most curves turn upward again at high gate voltage, so the two definitions differ.
- **Power, energy, delay.** These are fixed formulas of the supplied columns (verified to machine precision):

$$ \mathrm{SPD}=V_{DS}\,I_{off},\quad \mathrm{DED}=C_{eff}V_{DS}^2,\quad \tau_d=\frac{C_{eff}V_{DS}}{I_{on}},\quad P_{dyn}=\frac{\mathrm{DED}}{\tau_d}=V_{DS}\,I_{on},\qquad V_{DS}=0.05\ \mathrm V $$

So log10 SPD duplicates log10 Ioff, and P_dyn is nearly constant because Ion barely varies.

**Theory: subthreshold swing.** The swing is the gate voltage needed to cut the current tenfold:

$$ S = \left(\frac{\partial \log_{10} I}{\partial V}\right)^{-1}\ \text{(mV/decade)},\qquad S_{\text{Boltzmann}} = \frac{k_BT}{q}\ln 10 $$

The Boltzmann value is the limit for *thermionic* switching. The gate grid is coarse (0.1 V), so we interpolate the falling branch (from V = 0 to the curve minimum) with **PCHIP**, a piecewise cubic that never overshoots. We take the steepest slope.

**What is optimised.** Nothing is fitted here. The targets are deterministic transformations, and PCHIP is an interpolation, not a fit. Its node slopes are a weighted harmonic mean of the neighbouring secants, set to 0 at local extrema, which keeps it monotone wherever the data are:

$$ \frac{w_1+w_2}{d_k} = \frac{w_1}{\delta_{k-1}} + \frac{w_2}{\delta_k} $$

**Failure mode.** One grid step can drop several decades, so the swing depends on the interpolation method. It is reported only as a supporting result.

In [ ]:
curves = pd.read_csv(res / "curve_targets.csv")
curves[["log10_Ion_Ioff", "log10_Ion_Ioff_min", "log10_SPD", "log10_DED", "log10_tau_d", "log10_P_dyn",
        "swing_mV_per_dec", "swing_raw_mV_per_dec"]].describe().round(3)

**Figure: swing extraction on 12 sample curves** (blue = falling branch with its PCHIP fit, red dashed = steepest tangent, titles give the PCHIP swing, the raw-grid swing and the Boltzmann limit).

In [ ]:
show("swing_check.png")

## 3. Evaluation design: how we measure generalisation honestly

**What we do.** We split by **whole curves** and use four protocols:

| Protocol | Question it answers |
|---|---|
| supplied | interpolation: new temperatures *between* known ones, known layouts |
| temperature_forward | extrapolation in T: train ≤ 60 K, test 62–78 K |
| vacancy_block | unseen layouts *inside* the range: test 0.8–1.2 % (0.7 and 1.3 % dropped from training) |
| vacancy_extrapolation | unseen layouts *beyond* the range: train ≤ 1.4 % (1.5 % dropped), test 1.6–2.0 % |

**Theory.** We want the **generalisation error** $\mathbb E_{(x,y)\sim\mathcal D_{test}}\,\ell(y, \hat f(x))$ under the distribution we actually care about. Random row splits would put gate points of one sweep on both sides, and neighbouring vacancy levels are near-copies because of the nesting. Either would make test data almost identical to training data, i.e. *leakage*.

**What is optimised: hyperparameters by blocked cross-validation.** Inside each training pool, we choose the hyperparameters θ that minimise the CV error over K blocks of consecutive levels $B_k$, with an embargo $E_k$ (neighbouring levels removed from training):

$$ \hat\theta = \arg\min_\theta \frac1K\sum_{k=1}^K \frac{1}{|B_k|}\sum_{i\in B_k}\big(y_i - \hat f_\theta^{(-B_k\cup E_k)}(x_i)\big)^2 $$

The blocks run over vacancy for three protocols and over temperature for the 62–78 K test, so tuning resembles the final question. The final test set is evaluated once, after all choices are fixed.

**Failure mode.** Nested layouts mean the embargo only *reduces* leakage. With one layout, no split can show how a different layout behaves.

In [ ]:
s = pd.read_csv(res / "split_summary.csv")
s[s.protocol == s.split][["protocol", "train_curves", "test_curves", "train_T_K", "test_T_K", "test_vac_pct"]]

In [ ]:
s[s.protocol == "temperature_forward"][["split", "train_curves", "test_curves", "train_T_K", "test_T_K"]]   # temperature-blocked CV folds

## 4. Metrics: how we score a prediction

$$ \mathrm{MAE}=\frac1n\sum|y_i-\hat y_i|,\quad \mathrm{RMSE}=\sqrt{\tfrac1n\sum(y_i-\hat y_i)^2},\quad R^2 = 1-\frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2},\quad \text{skill}=1-\frac{\mathrm{MSE}_{model}}{\mathrm{MSE}_{best\ naive}} $$

- **MAE and RMSE** are in decades because targets are log10. An MAE of 1 means a factor-10 error on average.
- **$R^2 < 0$** means worse than predicting the test-set mean.
- **Skill** compares with the better of two naive baselines on the *same* test set. 0 means no better than the trivial guess, and negative means worse.

These are what we *report*. What each model *minimises* during training is given in its own section.

## 5. Naive baselines: the trivial guesses every model must beat

**What we do.** There are two baselines:
- **Per-vacancy-level mean:** predict the average of the training curves at the same, or nearest, vacancy level, ignoring temperature.
- **1-nearest neighbour:** copy the training curve closest in standardised (T, vacancy).

**Theory and what is optimised.** The group mean is the constant that minimises squared error within the group, $\arg\min_c \sum_{i\in g}(y_i-c)^2 = \bar y_g$. Nearest neighbour fits nothing: $\hat y(x)=y_{j^*}$ with $j^*=\arg\min_j\lVert z(x)-z(x_j)\rVert_2$, where $z$ is the standardised input.

**Why they matter here.** At a fixed layout the curves change slowly with temperature, so nearest neighbour is already strong for interpolation. Any claim that a model "learned something" must beat these.

## 6. Ridge and polynomial ridge regression (supervised learning: regularised linear models)

**What we do.** We fit a linear model of the standardised inputs, or of all their products up to degree $d$ (polynomial features), with a penalty on large coefficients.

**What is optimised.**

$$ \hat\beta = \arg\min_\beta\ \lVert y-\bar y - Z\beta\rVert_2^2 + \alpha\lVert\beta\rVert_2^2 \ \ \Longrightarrow\ \ \hat\beta=(Z^\top Z+\alpha I)^{-1}Z^\top(y-\bar y) $$

$Z$ is the standardised (and, for polynomial ridge, expanded) input matrix, and the intercept $\bar y$ is not penalised. The penalty $\alpha$ and the degree $d$ are hyperparameters, chosen by blocked CV (section 3).

**Theory: bias and variance.** Larger $\alpha$ shrinks the coefficients towards 0: more bias, less variance. The **effective degrees of freedom** count how many parameters the fit really uses:

$$ \mathrm{df}(\alpha)=\mathrm{tr}\big[Z(Z^\top Z+\alpha I)^{-1}Z^\top\big]=\sum_j \frac{d_j^2}{d_j^2+\alpha} $$

with $d_j$ the singular values of $Z$.

**Failure mode here.** The I–V curves drop by several decades within one gate step, and a low-degree polynomial cannot bend that fast. High degrees oscillate and extrapolate wildly.

**Check: ridge written from scratch.** `wte2ml/ridge_numpy.py` implements the formula above in NumPy. Below it is compared with the sklearn model the pipeline uses, on the real cubic-feature task.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from wte2ml.models import features
from wte2ml.ridge_numpy import fit_ridge, effective_dof, coefficient_path
poly = PolynomialFeatures(3, include_bias=False)
X = poly.fit_transform(features(curves, "vac_pct", None)); y = curves["log10_Ion_Ioff"].to_numpy()
for a in cfg["models"]["ridge"]["alpha"]:
    ours, sk = fit_ridge(X, y, a), make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(X, y)
    print(f"alpha={a:g}: max |prediction difference| = {np.abs(ours.predict(X) - sk.predict(X)).max():.1e}, effective dof = {effective_dof(X, a):.2f} of {X.shape[1]}")

In [ ]:
from wte2ml.plots import coefficient_path_lines
alphas = np.logspace(-4, 4, 60)
fig, ax = plt.subplots(figsize=(7, 4))
coefficient_path_lines(ax, alphas, coefficient_path(X, y, alphas), list(poly.get_feature_names_out(["T", "vac"])))
plt.show()

## 7. Gradient boosting (supervised learning: ensembles of trees)

**What we do.** We add many shallow regression trees one after another. Each new tree corrects what the sum so far still gets wrong. We use sklearn's histogram-based version, which bins the inputs for speed.

**What is optimised.** The squared loss over the training data, minimised by *functional gradient descent*:

$$ F_m(x)=F_{m-1}(x)+\nu\,h_m(x),\qquad h_m=\arg\min_{h\in\text{trees}}\sum_i\big(r_i^{(m)}-h(x_i)\big)^2,\qquad r_i^{(m)}=-\frac{\partial L(y_i,F)}{\partial F}\Big|_{F_{m-1}}=y_i-F_{m-1}(x_i) $$

For squared loss the negative gradient is just the residual. Each tree $h_m$ is grown greedily: every split is the one that reduces the squared error most. The learning rate $\nu$ shrinks each step. The tree depth and the number of trees are chosen by blocked CV.

**Why it fits here.** Trees represent steps and thresholds naturally, which suits sudden switching.

**Failure mode.** A tree predicts a constant in each region, so outside the training range (above 60 K, beyond 1.4 % vacancy) boosting repeats the values at the edge of the data and cannot follow a trend.

## 8. Gaussian-process regression (supervised learning: kernel methods with uncertainty)

**What we do.** We place a prior over smooth functions of (T, vacancy) and condition it on the training curves. For each prediction we get a mean *and* a standard deviation. For I–V curves we fit one GP per gate voltage, which keeps every kernel matrix at most 638 × 638.

**Theory.** $f\sim\mathcal{GP}(0,k)$ with kernel

$$ k(x,x') = \sigma_f^2\,\kappa\Big(\sqrt{\textstyle\sum_j (x_j-x'_j)^2/\ell_j^2}\Big) + \sigma_n^2\,\delta_{xx'} $$

where $\kappa$ is RBF ($e^{-r^2/2}$) or Matérn-5/2, there is one length scale $\ell_j$ per input, and $\sigma_n^2$ is white noise. The posterior at a new point $x_*$ is

$$ \mu(x_*)=k_*^\top(K+\sigma_n^2I)^{-1}y,\qquad \sigma^2(x_*)=k(x_*,x_*)-k_*^\top(K+\sigma_n^2I)^{-1}k_* $$

**What is optimised (two levels).**
1. *Inside each fit*, the kernel hyperparameters $\theta=(\sigma_f,\ell_j,\sigma_n)$ maximise the **log marginal likelihood**, which trades data fit against model complexity:

$$ \log p(y\mid X,\theta) = -\tfrac12 y^\top K_\theta^{-1}y - \tfrac12\log|K_\theta| - \tfrac n2\log 2\pi $$

2. *Across fits*, the kernel **family** (RBF vs. Matérn-5/2) is chosen by blocked CV.

**Why it fits here.** The data are small and smooth in temperature, and an error bar is valuable.

**Failure mode.** The fitted vacancy length scale is short, because neighbouring layouts genuinely differ. So for an unseen layout the GP reverts towards its mean, and its uncertainty does not grow enough (see section 14).

**Figure: sample I–V curves with the GP mean ± 2σ band, boosting and the MLP**, for each protocol at its lowest, middle and highest test vacancy level.

In [ ]:
show("iv_samples_gp_bands.png")

## 9. Multi-layer perceptron (neural networks)

**What we do.** We train a small fully connected network: 2–3 hidden layers of 64–128 units with the smooth SiLU activation $\sigma(z)=z/(1+e^{-z})$. For I–V curves it maps (T, vacancy) to all 20 gate points at once. We train 5 random initialisations (seeds) and report mean ± std.

**What is optimised.** The mean squared error on standardised targets, with decoupled weight decay (AdamW):

$$ \min_\theta\ \frac1n\sum_i\lVert y_i-f_\theta(x_i)\rVert^2,\qquad \theta_{t+1}=\theta_t-\eta\Big(\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}+\lambda\,\theta_t\Big) $$

$\hat m_t$ and $\hat v_t$ are Adam's bias-corrected running means of the gradient and the squared gradient, $\eta$ is the learning rate and $\lambda$ the weight decay.

**Regularisation.**
- *Early stopping:* training stops when the error on 15 % of training **curves** (whole curves, never single points) has not improved for a set number of epochs, and the best weights are kept.
- *Hyperparameters:* width, depth and weight decay are chosen by blocked CV.

**Why it could fit here.** A network can represent a sharp switching curve without its shape being fixed in advance.

**Failure mode.** With about 600 curves from one nested layout, the network fits layout-specific bumps and extrapolates them unpredictably, with large differences between seeds. Training longer does not fix this (section 15).

In [ ]:
mlp = pd.read_csv(res / "mlp_vs_classical.csv")
mlp[mlp.target.isin(["log10_I", "log10_Ion_Ioff"])][["protocol", "target", "mlp_MAE_mean", "mlp_MAE_std",
    "best_classical_model", "best_classical_MAE", "best_naive_MAE", "mlp_beats_best_classical"]].round(3)

## 10. Results: how the models compare

The tables below give the MAE in decades, with the best model per protocol in bold. Read them together with the protocol question in section 3: the supplied split tests interpolation, the vacancy tests test unseen layouts. The MLP row is the mean of its per-seed MAEs.

In [ ]:
show("model_comparison_log10_Ion_Ioff.png"); show("model_comparison_log10_I.png")

In [ ]:
comparison = pd.read_csv(res / "model_comparison.csv")
comparison[comparison.target == "log10_Ion_Ioff"].pivot(index="model", columns="protocol", values="skill").round(2)   # skill vs best naive

**Predicted vs true** for every protocol (rows) and model (columns). Points on the diagonal are perfect predictions.

In [ ]:
show("pred_vs_true_log10_Ion_Ioff.png")

**Where errors occur:** the absolute error of each test curve over the temperature × vacancy grid, then the MAE against vacancy and against temperature. On unseen layouts the errors form vertical stripes: they depend on the layout, not on temperature.

In [ ]:
show("error_map_log10_Ion_Ioff.png"); show("error_profiles_log10_Ion_Ioff.png")

In [ ]:
show("pred_vs_true_log10_I.png"); show("error_map_log10_I.png"); show("error_profiles_log10_I.png")

## 11. Principal component analysis (unsupervised learning: dimensionality reduction)

**What we do.** We treat each I–V curve as a point in 20 dimensions (one per gate voltage) and find the few directions along which curves differ most.

**What is optimised.** The first component is the unit direction that maximises the variance of the projected (centred) curves; each later one does the same, orthogonal to the previous ones:

$$ v_1=\arg\max_{\lVert v\rVert=1} \mathrm{Var}(\tilde Y v)=\arg\max_{\lVert v\rVert=1} v^\top \Sigma v \ \Rightarrow\ \Sigma v_j=\lambda_j v_j,\qquad \text{share}_j=\lambda_j/\textstyle\sum_i\lambda_i $$

Equivalently, PCA minimises the squared reconstruction error of the curves with $k$ components, and it is computed from the SVD $\tilde Y=USV^\top$. The columns share one unit (decades), so the data are centred but not rescaled.

**Failure mode.** PCA is linear. A drop that moves along the gate axis needs several components, so "components needed" overstates the physical complexity. It is fitted on all curves and is descriptive only.

In [ ]:
pd.read_csv(res / "pca_explained_variance.csv").head(6).round(4)

**Figure:** (a) explained variance, (b) the shapes of PC1 and PC2 over gate voltage, (c) the curves in PC1–PC2 space coloured by k-means cluster, with markers for the on/off-ratio classes.

In [ ]:
show("pca_clusters.png")

## 12. Clustering: k-means and Gaussian mixtures (unsupervised learning)

**What we do.** We group the devices in the space of the first PCA scores without telling the algorithm anything about vacancy or temperature. Then we compare the groups with on/off-ratio classes (< 4, 4–9, ≥ 9 decades; the 9-decade cut lies in a sparse region of the histogram, the 4-decade cut is our convention).

**What is optimised.**
- **k-means** minimises the within-cluster sum of squares, $\min_{\{C_j,\mu_j\}}\sum_j\sum_{i\in C_j}\lVert z_i-\mu_j\rVert^2$, by alternating two steps: assign each point to the nearest centre, then move each centre to the mean of its points (Lloyd's algorithm). It uses 10 random restarts.
- **The Gaussian mixture** maximises the likelihood $\sum_i\log\sum_j\pi_j\,\mathcal N(z_i\mid\mu_j,\Sigma_j)$ by expectation–maximisation. **BIC** $=-2\log L + p\log n$ penalises extra components.

**How we judge the result.**
- The **silhouette** $s_i=(b_i-a_i)/\max(a_i,b_i)$ ($a$: mean distance to its own cluster, $b$: to the nearest other one) measures separation.
- The **adjusted Rand index** measures agreement with the classes: 0 = chance, 1 = identical.

**Failure mode.** The curves are not blobs. Each layout traces a thin strand as temperature changes, so the clusters become bands of vacancy levels and no natural number of clusters stands out.

In [ ]:
pd.read_csv(res / "cluster_scores.csv")[["method", "k", "silhouette", "bic", "ari_vs_ion_ioff_classes"]].round(3)

In [ ]:
a = pd.read_csv(res / "cluster_assignments.csv"); pd.crosstab(a.vac_pct.round(1), a.kmeans_k3)   # which vacancy levels land in which cluster

## 13. Input-noise robustness

**What we do.** We fit each model once on clean data, then predict the test set again with Gaussian noise added to temperature and vacancy. The noise is measured in grid steps (2 K, 0.1 %), and we watch how the test error grows.

**Quantity measured.**

$$ \mathrm{MAE}(\sigma)=\mathbb E_\varepsilon\Big[\tfrac1n\sum_i\big|\hat f(x_i+\varepsilon_i)-y_i\big|\Big],\qquad \varepsilon_{ij}\sim\mathcal N\big(0,(\sigma\Delta_j)^2\big) $$

Nothing is optimised here; it is a stress test of models that are already fitted.

**Why it matters here.** It separates smooth learned physics from an exact look-up of the right device in a dense grid.

**Failure mode.** Vacancy is really a label for one layout, so noise on it partly means "the neighbouring layout". This is a test of the surrogate's behaviour, not a model of real measurement error.

In [ ]:
rob = pd.read_csv(res / "robustness_summary.csv")
display(rob.pivot_table(index=["protocol", "model"], columns="noise_fraction", values="MAE_mean").round(3))
show("robustness_noise.png")

## 14. How certain are the results? Cluster bootstrap and GP coverage

**What we do (bootstrap).** We redraw the test set many times and recompute the error. Curves of one layout are near-copies, so the resampling unit is a **whole vacancy level**:

$$ \mathrm{MAE}^{(b)}=\frac{\sum_k S_{\ell^{(b)}_k}}{\sum_k N_{\ell^{(b)}_k}},\qquad \text{95 \% CI} = [\text{2.5th},\ \text{97.5th percentile of } \mathrm{MAE}^{(b)}] $$

$S_\ell$ is the summed |error| and $N_\ell$ the number of rows at level ℓ.
- **Paired comparisons:** each model is compared with the best naive baseline on the *same* resampled levels.
- **Multiple testing:** with $m$ comparisons, the Bonferroni-adjusted interval uses level $1-0.05/m$.

Only the saved predictions are used; nothing is refitted.

**What we do (GP coverage).** If the GP's σ is calibrated, the truth falls inside ±1σ 68.3 % of the time and inside ±1.96σ 95 % of the time:

$$ \widehat{\mathrm{cov}}_z=\tfrac1n\sum_i \mathbb 1\big[|y_i-\mu_i|\le z\,\sigma_i\big] $$

**Failure mode.** The vacancy tests contain only 5 levels, so their intervals are coarse and wide. No bootstrap can recover the uncertainty from having just *one* layout.

In [ ]:
boot = pd.read_csv(res / "bootstrap_ci.csv")
best = comparison[comparison["rank"] == 1][["protocol", "target", "model"]]
best.merge(boot, on=["protocol", "target", "model"])[["target", "protocol", "model", "n_levels", "MAE", "MAE_lo", "MAE_hi"]].round(3)

In [ ]:
nb = boot[boot.model != boot.reference]
print("comparisons:", len(nb), "| significantly better than best naive (Bonferroni):", int((nb.diff_hi_bonf < 0).sum()),
      "| significantly worse:", int((nb.diff_lo_bonf > 0).sum()))

In [ ]:
from wte2ml.plots import coverage_bars
cover = pd.read_csv(res / "gp_coverage.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
coverage_bars(axes[0], cover, "log10_Ion_Ioff"); coverage_bars(axes[1], cover, "log10_I")
plt.tight_layout(); plt.show()

## 15. Would the neural network do better if trained longer?

**What we do.** Some curve-level MLP fits stopped at the 1,500-epoch limit, not by early stopping. We re-ran four of the most affected jobs with the limit at 8,000 and compared them seed by seed.

**Theory.** If the network were under-trained, the error would fall with more epochs. If it is limited by the data (one layout, extrapolation), more epochs mainly fit the training levels more closely and can make extrapolation worse.

In [ ]:
eps = pd.read_csv(res / "epoch_sensitivity.csv")
eps.groupby(["protocol", "target", "max_epochs"]).agg(MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
    seeds_hit_cap=("hit_cap", "sum"), last_epoch=("stop_epoch", "max")).round(3)

## 16. Summary and limitations

- **Temperature is learnable, a new defect layout is not.** Models that see a layout at some temperatures predict it well at others, including higher ones. No model reliably predicts a vacancy level whose layout it has not seen, because the data contain a single nested random layout.
- **More flexible is not better here.** The GP wins on known layouts. On unseen layouts the naive baselines and ridge are as good or better, and the MLP is the least reliable.
- **Uncertainty must be honest.** Error bars must resample whole layouts, and the GP's own intervals are trustworthy only on layouts it has seen.
- **Limitations:** one nested layout; a coarse gate grid; a non-self-consistent simulation, not an experiment; small vacancy test sets (5 levels).

The key numbers with their sources are collected in `docs/REPORT_NOTES.md`, the design choices with dates in `docs/DECISIONS.md`, and the module map and glossary in `docs/ARCHITECTURE.md`.